# inspect reactome + and merge with kegg

In [ ]:
import requests
import pandas as pd
import time
import os
import pickle

# ============================================================================
# 1. DOWNLOAD REACTOME INTERACTIONS FILE
# ============================================================================

def download_interactions_file(output_file='reactome_interactions_human.txt'):
    """
    Download human protein-protein interactions from Reactome
    """
    print("Downloading Reactome human protein-protein interactions...")
    
    url =  'https://download.reactome.org/96/interactors/reactome.homo_sapiens.interactions.tab-delimited.txt'

    
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        # Save to file
        with open(output_file, 'w') as f:
            f.write(response.text)
        
        # Parse into dataframe
        df = pd.read_csv(output_file, sep='\t')
        print(f"✓ Downloaded interactions file")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}\n")
        
        return df
    
    except Exception as e:
        print(f"✗ Error: {e}")
        return None


# ============================================================================
# 2. DOWNLOAD PATHWAY MAPPING FILES
# ============================================================================

def download_uniprot_to_pathways(output_file='uniprot_to_pathways.txt'):
    """
    Download UniProt to Pathways mapping (all levels)
    """
    print("Downloading UniProt to Pathways mapping...")
    
    url = 'https://download.reactome.org/96/UniProt2Reactome_All_Levels.txt'
    
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        with open(output_file, 'w') as f:
            f.write(response.text)
        
        df = pd.read_csv(output_file, sep='\t', header=None)
        df.columns = ['uniprot_id', 'reactome_id', 'url', 'pathway_name', 'evidence_code', 'species']
        
        print(f"✓ Downloaded UniProt to Pathways mapping")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}\n")
        
        return df
    
    except Exception as e:
        print(f"✗ Error: {e}")
        return None


def download_ensembl_to_pathways(output_file='ensembl_to_pathways.txt'):
    """
    Download ENSEMBL to Pathways mapping (all levels)
    """
    print("Downloading ENSEMBL to Pathways mapping...")
    
    url = 'https://download.reactome.org/96/Ensembl2Reactome_All_Levels.txt'
    
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        with open(output_file, 'w') as f:
            f.write(response.text)
        
        df = pd.read_csv(output_file, sep='\t', header=None)
        df.columns = ['ensembl_id', 'reactome_id', 'url', 'pathway_name', 'evidence_code', 'species']
        
        print(f"✓ Downloaded ENSEMBL to Pathways mapping")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}\n")
        
        return df
    
    except Exception as e:
        print(f"✗ Error: {e}")
        return None


def download_ncbi_to_pathways(output_file='ncbi_to_pathways.txt'):
    """
    Download NCBI Gene ID to Pathways mapping (all levels)
    """
    print("Downloading NCBI Gene ID to Pathways mapping...")
    
    url = 'https://download.reactome.org/96/NCBI2Reactome_All_Levels.txt'
    
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        with open(output_file, 'w') as f:
            f.write(response.text)
        
        df = pd.read_csv(output_file, sep='\t', header=None)
        df.columns = ['ncbi_gene_id', 'reactome_id', 'url', 'pathway_name', 'evidence_code', 'species']
        
        print(f"✓ Downloaded NCBI Gene ID to Pathways mapping")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}\n")
        
        return df
    
    except Exception as e:
        print(f"✗ Error: {e}")
        return None


# ============================================================================
# 3. DOWNLOAD PATHWAY INFORMATION
# ============================================================================

# Fix the pathway hierarchy download function
def download_pathway_hierarchy(output_file='pathway_hierarchy.txt'):
    """
    Download pathway hierarchy information
    """
    print("Downloading pathway hierarchy...")
    
    url = 'https://download.reactome.org/96/ReactomePathwaysRelation.txt'
    
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        with open(output_file, 'w') as f:
            f.write(response.text)
        
        # Read without assigning column names first
        df = pd.read_csv(output_file, sep='\t', header=None)
        
        # Assign correct number of columns based on actual data
        if df.shape[1] == 4:
            df.columns = ['parent_pathway_id', 'parent_pathway_name', 'child_pathway_id', 'child_pathway_name']
        elif df.shape[1] == 2:
            df.columns = ['parent_pathway_id', 'child_pathway_id']
        
        print(f"✓ Downloaded pathway hierarchy")
        print(f"  Shape: {df.shape}\n")
        
        return df
    
    except Exception as e:
        print(f"✗ Error: {e}")
        return None

print("Fixed function ready to use!")


def download_pathway_list(output_file='pathway_list.txt'):
    """
    Download complete list of pathways
    """
    print("Downloading complete pathway list...")
    
    url = 'https://download.reactome.org/96/ReactomePathways.txt'
    
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        with open(output_file, 'w') as f:
            f.write(response.text)
        
        df = pd.read_csv(output_file, sep='\t', header=None)
        df.columns = ['reactome_id', 'pathway_name', 'species']
        
        print(f"✓ Downloaded pathway list")
        print(f"  Shape: {df.shape}")
        print(f"  Unique species: {df['species'].nunique()}\n")
        
        return df
    
    except Exception as e:
        print(f"✗ Error: {e}")
        return None

# ============================================================================
# 4. MERGE INTERACTIONS WITH PATHWAY INFORMATION  (updated)
# ============================================================================

def merge_interactions_with_pathways(
    interactions_df,
    uniprot_pathways_df,
    kegg_pathways_df=None,       # NEW: optional KEGG dataframe
    multiplex_proteins=None
):
    """
    Merge interactions with pathway information.
    Tags each pathway with its source (Reactome / KEGG / both).

    Expected KEGG dataframe columns: ['uniprot_id', 'pathway_name']
    (add a 'source' column yourself if you build it from KEGG API data)
    """
    print("Merging interactions with pathway information...")

    # ── 1. Tag sources ───────────────────────────────────────────────────────
    reactome = uniprot_pathways_df[['uniprot_id', 'pathway_name']].copy()
    reactome['source'] = 'Reactome'

    if kegg_pathways_df is not None:
        kegg = kegg_pathways_df[['gene_name_uniprot', 'pathway_name']].copy()
        kegg['source'] = 'KEGG'
        kegg = kegg.rename(columns={'gene_name_uniprot': 'uniprot_id'})
        all_pathways = pd.concat([reactome, kegg], ignore_index=True)
    else:
        all_pathways = reactome

    # ── 2. Optionally filter to multiplex proteins ───────────────────────────
    if multiplex_proteins is not None:
        print(f"  Filtering to {len(multiplex_proteins):,} multiplex proteins...")

        interactions_df = interactions_df[
            interactions_df['# Interactor 1 uniprot id'].str.replace('uniprotkb:', '').isin(multiplex_proteins) &
            interactions_df['Interactor 2 uniprot id'].str.replace('uniprotkb:', '').isin(multiplex_proteins)
        ].copy()
        print(f"  Interactions after filter: {len(interactions_df):,}")

        all_pathways = all_pathways[
            all_pathways['uniprot_id'].isin(multiplex_proteins)
        ].copy()
        print(f"  Pathway mappings after filter: {len(all_pathways):,}")

    # ── 3. Clean UniProt IDs ─────────────────────────────────────────────────
    interactions_df['uniprot1'] = interactions_df['# Interactor 1 uniprot id'].str.replace('uniprotkb:', '')
    interactions_df['uniprot2'] = interactions_df['Interactor 2 uniprot id'].str.replace('uniprotkb:', '')

    # ── 4. Collapse multiple pathways/sources per protein into one row ────────
    # So the merge doesn't explode row count unexpectedly
    def collapse(df, id_col):
        return (
            df.groupby(id_col)
            .agg(
                pathway_names=('pathway_name', lambda x: '|'.join(sorted(set(x)))),
                sources=('source',        lambda x: '|'.join(sorted(set(x))))
            )
            .reset_index()
        )

    collapsed = collapse(all_pathways, 'uniprot_id')

    # ── 5. Merge for protein 1 ───────────────────────────────────────────────
    merged = interactions_df.merge(
        collapsed.rename(columns={
            'uniprot_id':     'uniprot_id_1',
            'pathway_names':  'pathway_protein1',
            'sources':        'source_protein1'
        }),
        left_on='uniprot1', right_on='uniprot_id_1', how='left'
    ).drop(columns='uniprot_id_1')

    # ── 6. Merge for protein 2 ───────────────────────────────────────────────
    merged = merged.merge(
        collapsed.rename(columns={
            'uniprot_id':     'uniprot_id_2',
            'pathway_names':  'pathway_protein2',
            'sources':        'source_protein2'
        }),
        left_on='uniprot2', right_on='uniprot_id_2', how='left'
    ).drop(columns='uniprot_id_2')

    # ── 7. Common pathway + combined source label ────────────────────────────
    def find_common_pathway(row):
        p1 = set(row['pathway_protein1'].split('|')) if pd.notna(row['pathway_protein1']) else set()
        p2 = set(row['pathway_protein2'].split('|')) if pd.notna(row['pathway_protein2']) else set()
        common = p1 & p2
        return '|'.join(sorted(common)) if common else None

    def combined_sources(row):
        s1 = set(row['source_protein1'].split('|')) if pd.notna(row['source_protein1']) else set()
        s2 = set(row['source_protein2'].split('|')) if pd.notna(row['source_protein2']) else set()
        combined = s1 | s2
        return '|'.join(sorted(combined)) if combined else None

    merged['common_pathway']  = merged.apply(find_common_pathway,  axis=1)
    merged['pathway_sources'] = merged.apply(combined_sources,      axis=1)

    # ── 8. Summary ───────────────────────────────────────────────────────────
    print(f"✓ Merged {len(merged):,} interactions with pathway information")
    print(f"  Interactions with common pathway: {merged['common_pathway'].notna().sum():,}")
    if kegg_pathways_df is not None:
        src_counts = merged['pathway_sources'].value_counts(dropna=False)
        print(f"\n  Source breakdown (pathway_sources column):")
        for src, cnt in src_counts.items():
            print(f"    {src}: {cnt:,}")
    print()

    return merged

In [ ]:
print("="*70)
print("REACTOME DATA DOWNLOAD")
print("="*70 + "\n")

# Step 1: Download interactions
print("STEP 1: Download Protein-Protein Interactions")
print("-"*70)
interactions_df = download_interactions_file()

# Step 2: Download pathway mappings
print("STEP 2: Download Pathway Mappings")
print("-"*70)
uniprot_pathways_df = download_uniprot_to_pathways()
ensembl_pathways_df = download_ensembl_to_pathways()
ncbi_pathways_df = download_ncbi_to_pathways()

# Step 3: Download pathway information
print("STEP 3: Download Pathway Information")
print("-"*70)
pathway_list_df = download_pathway_list()
pathway_hierarchy_df = download_pathway_hierarchy()



In [ ]:
import pandas as pd
import pickle
import os
import networkx as nx

def load_layers_from_folder(layers_dir):
    """Load all *_layer.pkl files from a folder."""
    layers = {}
    pkl_files = [f for f in os.listdir(layers_dir) if f.endswith('_layer.pkl')]
    
    if not pkl_files:
        raise FileNotFoundError(f"No *_layer.pkl files found in {layers_dir}")
    
    print(f"Found {len(pkl_files)} layer files")
    
    for fname in sorted(pkl_files):
        cell_type = fname.replace('_layer.pkl', '')
        fpath = os.path.join(layers_dir, fname)
        with open(fpath, 'rb') as f:
            layers[cell_type] = pickle.load(f)
    
    print(f"✓ Loaded {len(layers)} layers\n")
    return layers


def convert_layer_nodes_to_uniprot(layers, uniprot_to_symbol):
    """
    Convert all node labels in layers from gene symbols to UniProt IDs.
    
    Parameters
    ----------
    layers : dict of cell_type -> nx.Graph (nodes are gene symbols)
    uniprot_to_symbol : dict mapping uniprot_id -> gene_symbol
    
    Returns
    -------
    converted_layers : dict of cell_type -> nx.Graph (nodes are uniprot IDs)
    unmapped_genes : set of genes that couldn't be mapped
    """
    
    # Reverse the mapping: symbol -> uniprot
    symbol_to_uniprot = {v.upper(): k for k, v in uniprot_to_symbol.items()}
    
    converted_layers = {}
    unmapped_genes = set()
    total_mapped = 0
    
    for cell_type, layer in layers.items():
        node_mapping = {}
        for node in layer.nodes():
            symbol_upper = str(node).upper()
            if symbol_upper in symbol_to_uniprot:
                node_mapping[node] = symbol_to_uniprot[symbol_upper]
                total_mapped += 1
            else:
                unmapped_genes.add(node)
                node_mapping[node] = node  # keep original if not found
        
        converted_layer = nx.relabel_nodes(layer, node_mapping)
        converted_layers[cell_type] = converted_layer
        
        n_mapped = sum(1 for v in node_mapping.values() if v in symbol_to_uniprot.values())
        print(f"  {cell_type}: {len(node_mapping)} nodes → {n_mapped} mapped to UniProt")
    
    if unmapped_genes:
        print(f"\n⚠️  {len(unmapped_genes)} genes could not be mapped:")
        for gene in sorted(list(unmapped_genes)[:10]):
            print(f"    - {gene}")
        if len(unmapped_genes) > 10:
            print(f"    ... and {len(unmapped_genes) - 10} more")
    
    return converted_layers, unmapped_genes


def save_converted_layers(converted_layers, output_dir='layers_uniprot'):
    """Save converted layers to new folder."""
    os.makedirs(output_dir, exist_ok=True)
    
    for cell_type, layer in converted_layers.items():
        safe_name = cell_type.replace('/', '_').replace(' ', '_')
        fpath = os.path.join(output_dir, f"{safe_name}_layer.pkl")
        with open(fpath, 'wb') as f:
            pickle.dump(layer, f, protocol=4)
    
    print(f"✓ Saved {len(converted_layers)} converted layers to {output_dir}\n")
    return output_dir


def get_multiplex_proteins(converted_layers):
    """Get the set of all unique UniProt IDs across all layers."""
    proteins = set()
    for cell_type, layer in converted_layers.items():
        proteins.update(layer.nodes())
    print(f"✓ Found {len(proteins):,} unique proteins across {len(converted_layers)} layers\n")
    return proteins




In [ ]:

# Step 1: Load UniProt mapping
uniprot_mapping_csv = 'ppi_chloe/d_uniprot_to_symbol_20251120.csv'

print("Loading UniProt to gene symbol mapping...")
mapping_df = pd.read_csv('../../og_gene_set/ppi_chloe/d_uniprot_to_symbol_20251120.csv')
uniprot_to_symbol = dict(zip(mapping_df['uniprotid'], mapping_df['symbol']))
print(f"✓ Loaded mapping for {len(uniprot_to_symbol):,} UniProt IDs\n")


In [ ]:
layers_dir_1 = "../multiplex_ts/layers"
layers_dir_2 = "../multiplex_2025hdrg/layers"

layers_1 = load_layers_from_folder(layers_dir_1)
layers_2 = load_layers_from_folder(layers_dir_2)

# Merge — if same cell type exists in both, second folder wins
layers = {**layers_1, **layers_2}

print(f"Total layers: {len(layers)}")



In [ ]:

# Step 3: Convert to UniProt
print("Converting gene symbols → UniProt IDs...\n")
converted_layers, unmapped = convert_layer_nodes_to_uniprot(layers, uniprot_to_symbol)

# Step 4: Save converted layers
output_dir = save_converted_layers(converted_layers, output_dir='layers_uniprot')

# Step 5: Get unique proteins for filtering
multiplex_proteins = get_multiplex_proteins(converted_layers)

print("="*80)
print("CONVERSION COMPLETE")
print("="*80)
print(f"Converted layers saved to: {output_dir}")
print(f"Use 'multiplex_proteins' set in your merge script to filter interactions")

In [ ]:
kegg_pathways_df = pd.read_csv("kegg_pathways_with_names_symbols.csv")
kegg_pathways_df.head()

In [ ]:
# Step 4: Merge and create combined dataset

if interactions_df is not None and uniprot_pathways_df is not None:
    print("STEP 4: Merge Interactions with Pathway Information")
    print("-"*70)
    merged_df = merge_interactions_with_pathways(
        interactions_df,
        uniprot_pathways_df,
        kegg_pathways_df,
        multiplex_proteins=multiplex_proteins
    )


    # Save merged data
    merged_df.to_csv('reactome_interactions_with_pathways.csv', index=False)
    print(f"✓ Saved to reactome_interactions_with_pathways.csv\n")

# Summary
print("="*70)
print("SUMMARY")
print("="*70)
print(f"✓ Downloaded files:")
print(f"  - reactome_interactions_human.txt ({interactions_df.shape[0] if interactions_df is not None else 0} interactions)")
print(f"  - uniprot_to_pathways.txt ({uniprot_pathways_df.shape[0] if uniprot_pathways_df is not None else 0} mappings)")
print(f"  - ensembl_to_pathways.txt ({ensembl_pathways_df.shape[0] if ensembl_pathways_df is not None else 0} mappings)")
print(f"  - ncbi_to_pathways.txt ({ncbi_pathways_df.shape[0] if ncbi_pathways_df is not None else 0} mappings)")
print(f"  - pathway_list.txt ({pathway_list_df.shape[0] if pathway_list_df is not None else 0} pathways)")
print(f"  - pathway_hierarchy.txt ({pathway_hierarchy_df.shape[0] if pathway_hierarchy_df is not None else 0} relationships)")
print(f"  - reactome_interactions_with_pathways.csv (merged data)")

In [ ]:
merged_df.info()

In [ ]:
merged_df.head()